# Итоговый submission для kaggle

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
preds_IC50 = pd.read_csv("/Users/irinaryzova/Desktop/ДПО_МИФИ/Хакатон/Models_log_ag/preds_IC50_q50_q90.csv")
preds_CC50 = pd.read_csv("/Users/irinaryzova/Desktop/ДПО_МИФИ/Хакатон/Models_log_ag/preds_CC50_q50_q90.csv")
preds_SI = pd.read_csv("/Users/irinaryzova/Desktop/ДПО_МИФИ/Хакатон/Models_log_ag/preds_SI_q50_q90.csv")

In [3]:
preds_IC50.head()

,IC50_q50_mM,IC50_q90_mM
0,49.542484,8.238338
1,120.002686,26.569252
2,37.690804,4.589764
3,106.347330,14.660975
4,145.984710,23.088036


In [4]:
preds_CC50.head()

,CC50_q50_mM,CC50_q90_mM
0,118.83120,29.756618
1,317.37160,96.337090
2,280.03610,67.549706
3,269.49332,47.096000
4,324.34290,84.969740


In [5]:
preds_SI.head()

,SI_q10,SI_q50,SI_q90
0,1.014159,3.004862,25.684666
1,0.877321,3.334183,12.670206
2,2.221371,5.504696,21.327780
3,0.996908,2.597597,20.635807
4,0.990705,2.207856,7.295627


Для `SI` был дополнительно рассчитан смешанный прогноз `SI_q10_q50` как взвешенная комбинация нижнего и медианного квантилей. Такой подход использован для того, чтобы сохранить основную устойчивость центрального прогноза `q50`, но при этом немного скорректировать его в сторону нижней оценки `q10`.

С аналитической точки зрения это позволяет получить более осторожный прогноз по индексу селективности и снизить риск систематического завышения значений, если медианный квантиль оказывается слишком оптимистичным. Выбор весов `0.15` и `0.85` означает, что основное влияние сохраняется за `q50`, а `q10` используется как мягкая корректировка прогноза.

In [6]:
preds_SI['SI_q10_q50'] = 0.15 * preds_SI['SI_q10'] + 0.85 * preds_SI['SI_q50']
preds_SI.head()

,SI_q10,SI_q50,SI_q90,SI_q10_q50
0,1.014159,3.004862,25.684666,2.706256
1,0.877321,3.334183,12.670206,2.965654
2,2.221371,5.504696,21.327780,5.012197
3,0.996908,2.597597,20.635807,2.357494
4,0.990705,2.207856,7.295627,2.025284


In [7]:
preds_SI.describe()

,SI_q10,SI_q50,SI_q90,SI_q10_q50
count,250.000000,250.000000,250.000000,250.000000
mean,1.738752,12.981386,167.539227,11.294991
std,2.746323,33.445399,663.713072,28.819783
min,0.072237,1.153724,3.840761,1.128673
25%,0.990332,2.588324,17.112993,2.334721
50%,1.005349,3.671730,26.112457,3.262557
75%,1.155984,7.672009,48.852111,6.693471
max,20.539394,283.083340,5463.145000,243.027746


In [8]:
submission = pd.concat([preds_IC50['IC50_q50_mM'], preds_CC50['CC50_q50_mM'], preds_SI['SI_q10_q50']], axis=1)
submission.head()

,IC50_q50_mM,CC50_q50_mM,SI_q10_q50
0,49.542484,118.83120,2.706256
1,120.002686,317.37160,2.965654
2,37.690804,280.03610,5.012197
3,106.347330,269.49332,2.357494
4,145.984710,324.34290,2.025284


In [9]:
submission['index'] = submission.index
submission = submission[['index', 'IC50_q50_mM', 'CC50_q50_mM', 'SI_q10_q50']]
submission.head()

,index,IC50_q50_mM,CC50_q50_mM,SI_q10_q50
0,0,49.542484,118.83120,2.706256
1,1,120.002686,317.37160,2.965654
2,2,37.690804,280.03610,5.012197
3,3,106.347330,269.49332,2.357494
4,4,145.984710,324.34290,2.025284


In [10]:
submission.rename(columns={'IC50_q50_mM': 'IC50', 'CC50_q50_mM': 'CC50', 'SI_q10_q50': 'SI'}, inplace=True)
submission.head()

,index,IC50,CC50,SI
0,0,49.542484,118.83120,2.706256
1,1,120.002686,317.37160,2.965654
2,2,37.690804,280.03610,5.012197
3,3,106.347330,269.49332,2.357494
4,4,145.984710,324.34290,2.025284


In [11]:
submission.to_csv("/Users/irinaryzova/Desktop/ДПО_МИФИ/Хакатон/Models_log_ag/submission_q50_q90_1.csv", index=False)